In [1]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from huggingface_hub import snapshot_download  # если нужно скачать LoRA
import os

# Пути (вы уже задали)
CACHE_DIR = r'/home/jupyter/filestore/shared'
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Функция загрузки LoRA (без изменений, работает)
def load_lora(pipeline, lora_path: str, lora_scale: float = 0.8, fuse: bool = True):
    """
    Загрузка LoRA адаптера.
    - fuse=True: сливает веса в модель (постоянно) — проще, но нельзя выключить.
    - fuse=False: загружает, но не сливает — тогда при генерации нужно передавать scale через cross_attention_kwargs.
    """
    print(f"Loading LoRA: {lora_path}")
    try:
        pipeline.load_lora_weights(lora_path)
        if fuse:
            pipeline.fuse_lora(lora_scale=lora_scale)
            print(f"LoRA fused with scale {lora_scale}")
        else:
            # В таком случае при вызове pipeline нужно добавить cross_attention_kwargs={"scale": lora_scale}
            print(f"LoRA loaded (not fused). Use cross_attention_kwargs during generation.")
    except Exception as e:
        print(f"Error loading LoRA: {e}")
    return pipeline

# 2. Функция загрузки SDXL без ControlNet
def load_sdxl_pipeline(use_custom_vae: bool = True):
    """
    Загружает базовый SDXL пайплайн без ControlNet и IP-Adapter.
    """
    print("Loading SDXL pipeline...")
    
    vae = None
    if use_custom_vae:
        vae = AutoencoderKL.from_pretrained(
            "madebyollin/sdxl-vae-fp16-fix",
            torch_dtype=torch.float16,
            cache_dir=CACHE_DIR,
        )
        print("Loaded custom VAE")

    pipeline = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL,
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
        cache_dir=CACHE_DIR,
    )
    pipeline = pipeline.to(device)
    print("Pipeline ready.\n")
    return pipeline

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-07-04 07:49:30.117614: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
# Загружаем пайплайн
pipe = load_sdxl_pipeline()

Loading SDXL pipeline...
Loaded custom VAE


Loading pipeline components...: 100%|██████████| 7/7 [00:05<00:00,  1.19it/s]


In [ ]:
# Генерация
prompt = "v0ng44g, p14nt1ng, a painting of a cactus in desert in the style of van Gogh, in a desert, night, stars, by van Gogh"
negative_prompt = "txt, logo, duplicates, dupes, deforumed"

image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7,
        height=1024,
        width=1024,
        generator=torch.Generator(device="cuda").manual_seed(42)
    ).images[0]

image

In [ ]:
# Загружаем LoRA (например, Ван Гога)
lora_path = "/home/jupyter/filestore/shared/lora_weights/v0ng44g, p14nt1ng.safetensors"
pipe = load_lora(pipe, lora_path, lora_scale=0.8, fuse=True)

In [ ]:
# Генерация
prompt = "v0ng44g, p14nt1ng, a painting of a cactus in desert in the style of van Gogh, in a desert, night, stars, by van Gogh"
negative_prompt = "txt, logo, duplicates, dupes, deforumed"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7,
    height=1024,
    width=1024,
    generator=torch.Generator(device="cuda").manual_seed(42)
).images[0]

image